# VentriQ — Phase 4: ECG Waveform-to-Image Paper Renderer

**Objective:** Convert digital 12-lead ECG waveforms into photorealistic, standardized ECG paper recordings with standard clinical formatting, millimeter grid layout, lead labeling, and calibration pulses.

### Standard Cardiology Geometry & Layout:
- **Paper Speed:** $25\text{ mm/s}$ ($1\text{ mm} = 0.04\text{ s} = 40\text{ ms}$; $5\text{ mm} = 0.20\text{ s} = 200\text{ ms}$)
- **Voltage Gain:** $10\text{ mm/mV}$ ($1\text{ mm} = 0.1\text{ mV}$; $5\text{ mm} = 0.5\text{ mV}$; $10\text{ mm} = 1.0\text{ mV}$)
- **Standard 3x4 + 1 Rhythm Strip Layout:**
  - Row 0: Lead I (0–2.5s), aVR (2.5–5.0s), V1 (5.0–7.5s), V4 (7.5–10.0s)
  - Row 1: Lead II (0–2.5s), aVL (2.5–5.0s), V2 (5.0–7.5s), V5 (7.5–10.0s)
  - Row 2: Lead III (0–2.5s), aVF (2.5–5.0s), V3 (5.0–7.5s), V6 (7.5–10.0s)
  - Row 3: Lead II Continuous 10-Second Rhythm Strip (0.0–10.0s)
- **Calibration Pulse:** $1.0\text{ mV} (10\text{ mm}) \times 0.20\text{ s} (5\text{ mm})$ rectangular pulse at each lead row.
- **Amplitude Scaling Adaptation:** Maps EchoNext z-score normalized amplitudes safely to display millimeters without uncalibrated clipping.

In [ ]:
from pathlib import Path
import sys

# Ensure project root is in sys.path
NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.data.dataset import DatasetManifest, VentriQTorchDataset
from src.rendering import (
    ECGImage,
    ECGLayout,
    ECGLayoutConfig,
    ECGPaperRenderer,
    GridConfig,
    load_rendering_config,
)
from src.utils.paths import (
    FIGURES_DIR,
    REPORTS_DIR,
    SYNTHETIC_DATA_DIR,
    get_raw_echonext_dir,
)

print(f"Project Root: {PROJECT_ROOT}")
print(f"PyTorch Version: {torch.__version__}")

## 1. Inspect Clinical ECG Paper Geometry & Coordinates

In [ ]:
cfg = load_rendering_config()
layout = ECGLayout(cfg)

print(f"DPI: {cfg.dpi}")
print(f"Pixels per mm: {layout.px_per_mm:.3f} px/mm")
print(f"Canvas Dimensions: {layout.canvas_width_px} x {layout.canvas_height_px} px ({cfg.paper_width_mm} x {cfg.paper_height_mm} mm)")
print(f"Paper Speed: {cfg.paper_speed_mm_s} mm/s  |  Voltage Gain: {cfg.voltage_gain_mm_mv} mm/mV")

segments = layout.get_lead_segments()
print(f"\nComputed {len(segments)} Lead Segments for layout '{cfg.layout_format}':")
for s in segments:
    print(f"  Row {s.row}, Col {s.col}: Lead {s.lead_name:10s} | Time: {s.start_time_sec:4.1f}s - {s.end_time_sec:4.1f}s | Start X: {s.start_x_mm:5.1f}mm | Baseline Y: {s.baseline_y_mm:5.1f}mm")

## 2. Load Real Waveform and Render Clinical ECG Paper

In [ ]:
dataset = VentriQTorchDataset(root=get_raw_echonext_dir())
sample = dataset[0]

print(f"Record ID: {sample['record_id']}")
print(f"Patient:   {sample['patient_key']}")
print(f"Split:     {sample['split']}")
print(f"LVEF:      {sample['lvef'].item():.1f}%")
print(f"LVEF <=45: {sample['lvef_lte_45'].item()}")

renderer = ECGPaperRenderer(config=cfg, supersample=1)
record_dict = {
    "record_id": sample["record_id"],
    "patient_key": sample["patient_key"],
    "split": sample["split"],
    "lvef": sample["lvef"].item(),
    "lvef_lte_45": sample["lvef_lte_45"].item(),
}

ecg_image = renderer.render(sample["waveform"], record=record_dict)
print(f"\nRendered Image Size: {ecg_image.size} (W x H)")

# Display inline with matplotlib
plt.figure(figsize=(14, 9), dpi=120)
plt.imshow(ecg_image.to_numpy())
plt.axis("off")
plt.title(f"VentriQ Clinical ECG Paper: {sample['record_id']} (LVEF: {sample['lvef'].item():.1f}%)", fontsize=12, pad=10)
plt.show()

## 3. Compare Preserved Normal LVEF vs Reduced LVEF Recordings

In [ ]:
# Find one preserved LVEF (>55%) and one reduced LVEF (<=45%)
normal_sample = next(s for s in dataset if s["lvef"].item() >= 55.0)
reduced_sample = next(s for s in dataset if s["lvef"].item() <= 45.0)

img_normal = renderer.render(normal_sample["waveform"], record=normal_sample)
img_reduced = renderer.render(reduced_sample["waveform"], record=reduced_sample)

fig, axes = plt.subplots(2, 1, figsize=(14, 16), dpi=100)
axes[0].imshow(img_normal.to_numpy())
axes[0].axis("off")
axes[0].set_title(f"Preserved Systolic Function: {normal_sample['record_id']} (LVEF = {normal_sample['lvef'].item():.1f}%)", fontsize=11)

axes[1].imshow(img_reduced.to_numpy())
axes[1].axis("off")
axes[1].set_title(f"Reduced Systolic Function: {reduced_sample['record_id']} (LVEF = {reduced_sample['lvef'].item():.1f}%)", fontsize=11)
plt.tight_layout()
plt.show()

## 4. Explore Clinical Grid Style Presets

In [ ]:
styles = ["standard_pink", "classic_red", "subtle_gray", "clinical_green"]
fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=100)

for ax, style_name in zip(axes.flat, styles):
    custom_grid = GridConfig.from_preset(style_name)
    custom_cfg = ECGLayoutConfig(dpi=100, grid=custom_grid)
    custom_renderer = ECGPaperRenderer(config=custom_cfg)
    style_img = custom_renderer.render(sample["waveform"], record=sample)
    
    ax.imshow(style_img.to_numpy())
    ax.axis("off")
    ax.set_title(f"Grid Preset: '{style_name}'", fontsize=10)

plt.tight_layout()
plt.show()

## 5. Verify Batch Image Manifest & Label Traceability

In [ ]:
manifest_csv = SYNTHETIC_DATA_DIR / "image_manifest.csv"
if manifest_csv.exists():
    df_manifest = pd.read_csv(manifest_csv)
    print(f"Loaded Image Manifest with {len(df_manifest)} records:")
    display_cols = ["record_id", "patient_key", "split", "lvef_value", "lvef_lte_45_flag", "shd_flag", "image_width", "image_height", "dpi"]
    print(df_manifest[display_cols].head(10).to_string(index=False))
    
    print("\nSplit Distribution in Synthetic Manifest:")
    print(df_manifest["split"].value_counts().to_string())
else:
    print("Run scripts/generate_ecg_images.py to create data/synthetic/image_manifest.csv")

## 6. PyTorch DataLoader Zero-Copy Image Pipeline Integration

In [ ]:
# Pass renderer directly as PyTorch transform
fast_renderer = ECGPaperRenderer(config=ECGLayoutConfig(dpi=100))
vision_dataset = VentriQTorchDataset(
    root=get_raw_echonext_dir(),
    split="train",
    transform=fast_renderer,
)

dataloader = DataLoader(vision_dataset, batch_size=4, shuffle=True)
batch = next(iter(dataloader))

print(f"Batched Image Tensor Shape: {batch['waveform'].shape} (B x C x H x W)")
print(f"Image Value Range:          [{batch['waveform'].min():.3f}, {batch['waveform'].max():.3f}]")
print(f"Batched LVEF Targets:       {batch['lvef'].tolist()}")
print(f"Batched LVEF <=45 Labels:   {batch['lvef_lte_45'].tolist()}")
print("\nPhase 4 Image Pipeline seamlessly feeds PyTorch vision backbones!")